# Supplementary figures: Count likelihood and detection diagnostics

Run after the training and evaluation commands in `bash/paper/`. SCENE outputs use the `scLDM` names referenced below.


In [ ]:
from pathlib import Path
import os
import json
import numpy as np
import pandas as pd

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "environment_scene.yaml").exists())
os.chdir(PROJECT_ROOT / "notebooks" / "figures")
for folder in ("fig_1", "fig_2", "fig_3", "fig_4", "fig_5", "fig_6", "app", "qc"):
    Path("output", folder).mkdir(parents=True, exist_ok=True)


# SCENE zero-inflation comparison

Compare recovered held-out counts for the SCENE-ZIP and SCENE-Poisson imputation runs.

In [ ]:
from pathlib import Path
import json

import h5py
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = PROJECT_ROOT
ADATA_PATH = PROJECT_ROOT / "data/pbmc_cite_seq.h5ad"
ZIP_DIR = PROJECT_ROOT / "results/imputation/pbmc_cite_seq_SCENE_zip"
POISSON_DIR = PROJECT_ROOT / "results/imputation/pbmc_cite_seq_SCENE_poisson"
OUTPUT_DIR = PROJECT_ROOT / "notebooks/figures/output/app"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 6,
    "axes.titlesize": 7,
    "axes.labelsize": 6,
    "xtick.labelsize": 5.5,
    "ytick.labelsize": 5.5,
    "legend.fontsize": 5.5,
    "axes.linewidth": 0.6,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.major.size": 2.5,
    "ytick.major.size": 2.5,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

In [ ]:
def load_heldout(path):
    with np.load(path) as heldout:
        col_key = "columns" if "columns" in heldout.files else "col"
        row_idx = heldout["row"].astype(np.int64, copy=False)
        col_idx = heldout[col_key].astype(np.int64, copy=False)
        true_count = heldout["heldout_count"].astype(np.float32, copy=False)
        n_obs = int(heldout["n_obs"]) if "n_obs" in heldout.files else None
        n_vars = int(heldout["n_vars"]) if "n_vars" in heldout.files else None
    return row_idx, col_idx, true_count, n_obs, n_vars


def load_h5ad_index_info(path):
    with h5py.File(path, "r") as handle:
        obs_index_key = handle["obs"].attrs.get("_index", "_index")
        var_index_key = handle["var"].attrs.get("_index", "_index")
        obs_names = handle["obs"][obs_index_key].asstr()[()]
        var_names = handle["var"][var_index_key].asstr()[()]
    return {
        "n_obs": obs_names.shape[0],
        "n_vars": var_names.shape[0],
        "obs_names": obs_names,
        "var_names": var_names,
    }


def validate_heldout_against_adata(adata_info, row_idx, col_idx, n_obs=None, n_vars=None):
    if n_obs is not None and n_obs != adata_info["n_obs"]:
        raise ValueError(f"Held-out n_obs={n_obs} does not match adata n_obs={adata_info['n_obs']}")
    if n_vars is not None and n_vars != adata_info["n_vars"]:
        raise ValueError(f"Held-out n_vars={n_vars} does not match adata n_vars={adata_info['n_vars']}")
    if row_idx.max(initial=-1) >= adata_info["n_obs"]:
        raise ValueError("Held-out row indices exceed adata n_obs")
    if col_idx.max(initial=-1) >= adata_info["n_vars"]:
        raise ValueError("Held-out column indices exceed adata n_vars")


def validate_saved_names(result_dir, adata_info, out_emb="SCLDM"):
    cell_names_path = result_dir / f"{out_emb}_cell_names.npy"
    gene_names_path = result_dir / f"{out_emb}_gene_names.npy"

    if cell_names_path.exists():
        saved = np.load(cell_names_path, allow_pickle=False)
        current = adata_info["obs_names"].astype(str)
        if not np.array_equal(saved, current):
            raise ValueError(f"Saved cell names in {result_dir.name} do not match adata obs names")

    if gene_names_path.exists():
        saved = np.load(gene_names_path, allow_pickle=False)
        current = adata_info["var_names"].astype(str)
        if not np.array_equal(saved, current):
            raise ValueError(f"Saved gene names in {result_dir.name} do not match adata var names")


def load_edge_lambda(
    result_dir,
    row_idx,
    col_idx,
    out_emb="SCLDM",
    chunk_size=2_000_000,
    allow_prediction_fallback=False,
):
    """Compute lambda for held-out cell-gene pairs without materializing the full matrix."""
    required = {
        "z_cells": result_dir / f"{out_emb}_cell_latent.npy",
        "z_genes": result_dir / f"{out_emb}_gene_latent.npy",
        "re_cells": result_dir / f"{out_emb}_re_cell.npy",
        "re_genes": result_dir / f"{out_emb}_re_gene.npy",
    }
    missing = [path.name for path in required.values() if not path.exists()]

    if missing:
        if allow_prediction_fallback:
            predictions_path = result_dir / "model_predictions.npz"
            with np.load(predictions_path) as predictions:
                if out_emb not in predictions.files:
                    raise KeyError(f"{out_emb!r} is missing from {predictions_path}")
                prediction = predictions[out_emb].astype(np.float32, copy=False)
            if prediction.shape[0] != row_idx.shape[0]:
                raise ValueError("Saved prediction length does not match held-out indices")
            return prediction, "model_predictions.npz fallback"
        raise FileNotFoundError(f"Missing files in {result_dir}: {missing}")

    z_cells = np.load(required["z_cells"], mmap_mode="r")
    z_genes = np.load(required["z_genes"], mmap_mode="r")
    re_cells = np.load(required["re_cells"], mmap_mode="r")
    re_genes = np.load(required["re_genes"], mmap_mode="r")

    lambdas = np.empty(row_idx.shape[0], dtype=np.float32)
    for start in range(0, row_idx.shape[0], chunk_size):
        end = min(start + chunk_size, row_idx.shape[0])
        rows = row_idx[start:end]
        cols = col_idx[start:end]

        zc = np.asarray(z_cells[rows], dtype=np.float32)
        zg = np.asarray(z_genes[cols], dtype=np.float32)
        rec = np.asarray(re_cells[rows], dtype=np.float32)
        reg = np.asarray(re_genes[cols], dtype=np.float32)

        dist = np.linalg.norm(zc - zg, axis=1)
        lambdas[start:end] = np.exp(rec + reg - dist)

    return lambdas, "reconstructed from saved latent/random-effect arrays"


def zero_truncated_poisson_mean(lam):
    lam = np.asarray(lam, dtype=np.float32)
    small = lam < 1e-6
    out = np.empty_like(lam, dtype=np.float32)
    out[small] = 1.0 + lam[small] / 2.0
    out[~small] = lam[~small] / (-np.expm1(-lam[~small]))
    return out


def load_scldm_alpha(result_dir, out_emb="SCLDM"):
    params_path = result_dir / f"{out_emb}_params.json"
    with open(params_path) as handle:
        params = json.load(handle)
    return float(params["alpha"])


def sigmoid(x):
    x = np.asarray(x, dtype=np.float32)
    out = np.empty_like(x, dtype=np.float32)
    positive = x >= 0
    out[positive] = 1.0 / (1.0 + np.exp(-x[positive]))
    exp_x = np.exp(x[~positive])
    out[~positive] = exp_x / (1.0 + exp_x)
    return out


def zip_nonzero_probability(lam, alpha):
    lam = np.clip(np.asarray(lam, dtype=np.float32), np.finfo(np.float32).tiny, None)
    return sigmoid(alpha * np.log(lam))


def mean_poisson_deviance_manual(x, mu):
    x = np.asarray(x).ravel()
    mu = np.asarray(mu).ravel()
    mu = np.clip(mu, np.finfo(float).tiny, None)

    terms = mu.copy()
    positive = x > 0
    terms[positive] = (
        x[positive] * np.log(x[positive] / mu[positive])
        - (x[positive] - mu[positive])
    )
    return 2 * np.mean(terms)

In [ ]:
zip_rows, zip_cols, zip_true, zip_n_obs, zip_n_vars = load_heldout(ZIP_DIR / "held_out_counts.npz")
poisson_rows, poisson_cols, poisson_true, poisson_n_obs, poisson_n_vars = load_heldout(POISSON_DIR / "held_out_counts.npz")

if not np.array_equal(zip_rows, poisson_rows):
    raise ValueError("ZIP and Poisson held-out row indices differ")
if not np.array_equal(zip_cols, poisson_cols):
    raise ValueError("ZIP and Poisson held-out column indices differ")
if not np.array_equal(zip_true, poisson_true):
    raise ValueError("ZIP and Poisson held-out counts differ")

adata_info = load_h5ad_index_info(ADATA_PATH)
validate_heldout_against_adata(adata_info, zip_rows, zip_cols, zip_n_obs, zip_n_vars)
validate_heldout_against_adata(adata_info, poisson_rows, poisson_cols, poisson_n_obs, poisson_n_vars)
validate_saved_names(ZIP_DIR, adata_info)
validate_saved_names(POISSON_DIR, adata_info)

row_idx = zip_rows
col_idx = zip_cols
true_vals = zip_true

print(f"Loaded {true_vals.shape[0]:,} held-out nonzero counts")
print(f"adata shape: {adata_info['n_obs']:,} cells x {adata_info['n_vars']:,} genes")

In [ ]:
lambda_zip, zip_source = load_edge_lambda(ZIP_DIR, row_idx, col_idx)
lambda_poisson, poisson_source = load_edge_lambda(
    POISSON_DIR,
    row_idx,
    col_idx,
    allow_prediction_fallback=True,
)

alpha_zip = load_scldm_alpha(ZIP_DIR)
pi_zip = zip_nonzero_probability(lambda_zip, alpha_zip)
ztp_zip = zero_truncated_poisson_mean(lambda_zip)
ztp_poisson = zero_truncated_poisson_mean(lambda_poisson)

predictions_by_mode = {
    "unknown_positive": {
        "title": "Unconditioned count recovery E[X]",
        "zip_label": "SCENE-ZIP: E[X]",
        "poisson_label": "SCENE-Poisson: E[X]",
        "zip": pi_zip * ztp_zip,
        "poisson": lambda_poisson,
        "output": "app_zero_inflation_unknown_positive.svg",
    },
    "conditioned_positive": {
        "title": "Conditioned count recovery E[X | X > 0]",
        "zip_label": "SCENE-ZIP: E[X | X > 0]",
        "poisson_label": "SCENE-Poisson: E[X | X > 0]",
        "zip": ztp_zip,
        "poisson": ztp_poisson,
        "output": "app_zero_inflation_conditioned_positive.svg",
    },
}

imputed_scldm_unknown_positive = predictions_by_mode["unknown_positive"]["zip"]
imputed_scldm_poisson_unknown_positive = predictions_by_mode["unknown_positive"]["poisson"]
imputed_scldm_conditioned_positive = predictions_by_mode["conditioned_positive"]["zip"]
imputed_scldm_poisson_conditioned_positive = predictions_by_mode["conditioned_positive"]["poisson"]

# Arrays used by the scatter and KDE plots below.
imputed_scldm = imputed_scldm_conditioned_positive
imputed_scldm_poisson = imputed_scldm_poisson_conditioned_positive

print(f"SCENE-ZIP lambda source: {zip_source}")
print(f"SCENE-ZIP alpha: {alpha_zip:.4f}")
print(f"SCENE-Poisson lambda source: {poisson_source}")

In [ ]:
x_true_all = np.asarray(true_vals).ravel()
analysis_results = {}
summary_rows = []

for mode_name, mode in predictions_by_mode.items():
    mode_true = x_true_all
    mode_zip = np.asarray(mode["zip"]).ravel()
    mode_poisson = np.asarray(mode["poisson"]).ravel()

    mask = (
        np.isfinite(mode_true)
        & np.isfinite(mode_zip)
        & np.isfinite(mode_poisson)
    )

    mode_true = mode_true[mask]
    mode_zip = mode_zip[mask]
    mode_poisson = mode_poisson[mask]

    dev_zip = mean_poisson_deviance_manual(mode_true, mode_zip)
    dev_poisson = mean_poisson_deviance_manual(mode_true, mode_poisson)

    analysis_results[mode_name] = {
        **mode,
        "x_true": mode_true,
        "pred_zip": mode_zip,
        "pred_poisson": mode_poisson,
        "dev_zip": dev_zip,
        "dev_poisson": dev_poisson,
    }

    summary_rows.extend([
        {
            "mode": mode_name,
            "model": "SCENE-ZIP",
            "prediction": mode["zip_label"],
            "mean_poisson_deviance": dev_zip,
            "n_heldout": mode_true.shape[0],
        },
        {
            "mode": mode_name,
            "model": "SCENE-Poisson",
            "prediction": mode["poisson_label"],
            "mean_poisson_deviance": dev_poisson,
            "n_heldout": mode_true.shape[0],
        },
    ])

summary = pd.DataFrame(summary_rows)

# Arrays used by the scatter and KDE plots below.
conditioned = analysis_results["conditioned_positive"]
x_true = conditioned["x_true"]
pred_zip = conditioned["pred_zip"]
pred_poisson = conditioned["pred_poisson"]
dev_zip = conditioned["dev_zip"]
dev_poisson = conditioned["dev_poisson"]

for row in summary.itertuples(index=False):
    print(f"{row.mode} | {row.model} mean Poisson deviance: {row.mean_poisson_deviance:.4f}")

summary

In [ ]:
SCATTER_MAX_POINTS = 100_000_000
rng = np.random.default_rng(42)

if x_true.shape[0] > SCATTER_MAX_POINTS:
    scatter_idx = rng.choice(x_true.shape[0], size=SCATTER_MAX_POINTS, replace=False)
else:
    scatter_idx = np.arange(x_true.shape[0])

scatter_true = x_true[scatter_idx]
scatter_preds = {
    "SCENE-ZIP": pred_zip[scatter_idx],
    "SCENE-Poisson": pred_poisson[scatter_idx],
}

red = "#D76155"
orange = "#E69F00"

model_order = ["SCENE-ZIP", "SCENE-Poisson"]
model_labels = {
    "SCENE-ZIP": "SCENE-ZIP",
    "SCENE-Poisson": "SCENE-Poisson",
}
model_colors = {
    "SCENE-ZIP": red,
    "SCENE-Poisson": orange,
}

maxv = np.percentile(
    np.concatenate([scatter_true, scatter_preds["SCENE-ZIP"], scatter_preds["SCENE-Poisson"]]),
    100,
)
maxv = max(2.0, float(np.ceil(maxv)))
minv = 1.0

fig, axes = plt.subplots(
    1,
    2,
    figsize=(7, 2.3),
    sharex=True,
    sharey=True,
)

for i, model in enumerate(model_order):
    ax = axes[i]
    ax.scatter(
        scatter_true,
        scatter_preds[model],
        s=3,
        alpha=0.35,
        color=model_colors[model],
        rasterized=True,
    )
    ax.plot([minv, maxv], [minv, maxv], "--", linewidth=1, color="gray")
    ax.set_xlim(minv, maxv)
    ax.set_ylim(minv, maxv)
    ax.set_xlabel("True counts")
    ax.set_xscale("log")
    ax.set_yscale("log")
    if i == 0:
        ax.set_ylabel("Imputed counts")
    else:
        plt.setp(ax.get_yticklabels(), visible=False)
    ax.set_title(f"True vs Imputed ({model_labels[model]})")

plt.tight_layout()
# fig.savefig(PROJECT_ROOT / "notebooks/figures/output/app/app_zero_inflation_scatter.svg", dpi=600, bbox_inches="tight")
plt.show()

In [ ]:
def plot_binned_comparison(mode_name, result, max_val=15):
    max_int = int(np.ceil(max_val))
    bins = np.arange(-0.5, max_int + 1.5, 1)

    red = "#D76155"      # SCENE-ZIP
    orange = "#4C9A8A"  # SCENE-Poisson
    gray = "0.35"

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(7, 2.3),
        sharex=True,
        sharey=True,
    )

    ax = axes[0]
    ax.hist(
        result["x_true"],
        bins=bins,
        density=False,
        alpha=0.45,
        color=gray,
        label="True held-out counts",
    )
    ax.hist(
        result["pred_zip"],
        bins=bins,
        density=False,
        alpha=0.55,
        color=red,
        label=result["zip_label"],
    )
    ax.set_title(f"SCENE-ZIP\nMean Poisson dev. = {result['dev_zip']:.3f}")
    ax.set_xlabel("Count / expected count")
    ax.set_ylabel("Frequency")
    ax.set_yscale("log")
    ax.set_xticks(np.arange(0, max_int + 1, 1))
    ax.legend(frameon=False)

    ax = axes[1]
    ax.hist(
        result["x_true"],
        bins=bins,
        density=False,
        alpha=0.45,
        color=gray,
        label="True held-out counts",
    )
    ax.hist(
        result["pred_poisson"],
        bins=bins,
        density=False,
        alpha=0.55,
        color=orange,
        label=result["poisson_label"],
    )
    ax.set_title(f"SCENE-Poisson\nMean Poisson dev. = {result['dev_poisson']:.3f}")
    ax.set_xlabel("Count / expected count")
    ax.set_yscale("log")
    ax.set_xticks(np.arange(0, max_int + 1, 1))
    ax.legend(frameon=False)

    fig.suptitle(result["title"], y=1.03, fontsize=7)
    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / result["output"], dpi=600, bbox_inches="tight")
    plt.show()


plot_binned_comparison("unknown_positive", analysis_results["unknown_positive"])
plot_binned_comparison("conditioned_positive", analysis_results["conditioned_positive"])

In [ ]:
from scipy.stats import gaussian_kde

KDE_MAX_POINTS = 100_000
rng = np.random.default_rng(42)


def sample_for_kde(values, max_points=KDE_MAX_POINTS):
    values = np.asarray(values).ravel()
    values = values[np.isfinite(values) & (values >= 0)]
    if values.shape[0] <= max_points:
        return values
    idx = rng.choice(values.shape[0], size=max_points, replace=False)
    return values[idx]


display_sample = np.concatenate([
    sample_for_kde(x_true, 100_000),
    sample_for_kde(pred_zip, 100_000),
    sample_for_kde(pred_poisson, 100_000),
])
max_val = np.percentile(display_sample, 99.5)
max_int = int(np.ceil(max_val))

count_bins = np.arange(-0.5, max_int + 1.5, 1)
grid = np.linspace(0, max_val, 500)

kde_zip = gaussian_kde(sample_for_kde(pred_zip))
kde_poisson = gaussian_kde(sample_for_kde(pred_poisson))

zip_density = kde_zip(grid)
poisson_density = kde_poisson(grid)

red = "#D76155"
orange = "#4C9A8A"
gray = "0.35"

fig, axes = plt.subplots(
    1,
    2,
    figsize=(7, 2.3),
    sharex=True,
    sharey=True,
)

ax = axes[0]
ax.hist(
    x_true,
    bins=count_bins,
    density=True,
    alpha=0.35,
    color=gray,
    label="True held-out counts",
)
ax.plot(
    grid,
    zip_density,
    color=red,
    linewidth=1.5,
    label="SCENE-ZIP prediction density",
)
ax.set_title("SCENE-ZIP")
ax.set_xlabel("Count / expected count")
ax.set_ylabel("Density")
ax.set_yscale("log")
ax.set_xticks(np.arange(0, max_int + 1, 1))
ax.legend(frameon=False)

ax = axes[1]
ax.hist(
    x_true,
    bins=count_bins,
    density=True,
    alpha=0.35,
    color=gray,
    label="True held-out counts",
)
ax.plot(
    grid,
    poisson_density,
    color=orange,
    linewidth=1.5,
    label="SCENE-Poisson prediction density",
)
ax.set_title("SCENE-Poisson")
ax.set_xlabel("Count / expected count")
ax.set_yscale("log")
ax.set_xticks(np.arange(0, max_int + 1, 1))
ax.legend(frameon=False)

plt.tight_layout()
# fig.savefig(PROJECT_ROOT / "notebooks/figures/output/app_zero_inflation_smooth_density.svg", dpi=600, bbox_inches="tight")
plt.show()

In [ ]:
# ZIP detection-vs-intensity diagnostic over the full observed matrix.
# This streams cell blocks and never materializes dense X, full cdist, lambda, or pi matrices.

ZERO_RATE_CELL_CHUNK_SIZE = 512
ZERO_RATE_RANGE_HIST_BINS = 4_000
ZERO_RATE_PLOT_BINS = 40
ZERO_RATE_LOWER_Q = 0.005
ZERO_RATE_UPPER_Q = 0.995


def iter_zip_log10_lambda_blocks(result_dir, cell_chunk_size=ZERO_RATE_CELL_CHUNK_SIZE):
    z_cells = np.load(result_dir / "SCLDM_cell_latent.npy", mmap_mode="r")
    z_genes = np.load(result_dir / "SCLDM_gene_latent.npy", mmap_mode="r")
    re_cells = np.load(result_dir / "SCLDM_re_cell.npy", mmap_mode="r").reshape(-1)
    re_genes = np.load(result_dir / "SCLDM_re_gene.npy", mmap_mode="r").reshape(-1)

    z_genes_arr = np.asarray(z_genes, dtype=np.float32)
    z_genes_t = z_genes_arr.T
    z_genes_norm = np.sum(z_genes_arr * z_genes_arr, axis=1, dtype=np.float32)
    re_genes_arr = np.asarray(re_genes, dtype=np.float32)

    n_cells = z_cells.shape[0]
    log10_e = np.float32(1.0 / np.log(10.0))

    for start in range(0, n_cells, cell_chunk_size):
        end = min(start + cell_chunk_size, n_cells)
        zc = np.asarray(z_cells[start:end], dtype=np.float32)
        rec = np.asarray(re_cells[start:end], dtype=np.float32)

        block = zc @ z_genes_t
        block *= -2.0
        block += np.sum(zc * zc, axis=1, dtype=np.float32)[:, None]
        block += z_genes_norm[None, :]
        np.maximum(block, 0.0, out=block)
        np.sqrt(block, out=block)

        block *= -1.0
        block += rec[:, None]
        block += re_genes_arr[None, :]
        block *= log10_e

        yield start, end, block


def histogram_quantile(hist, edges, q):
    cumulative = np.cumsum(hist, dtype=np.float64)
    target = q * cumulative[-1]
    idx = int(np.searchsorted(cumulative, target, side="left"))
    idx = min(max(idx, 0), hist.shape[0] - 1)
    prev = cumulative[idx - 1] if idx > 0 else 0.0
    denom = hist[idx]
    frac = 0.0 if denom == 0 else (target - prev) / denom
    return edges[idx] + frac * (edges[idx + 1] - edges[idx])


def full_matrix_zero_rate_by_zip_intensity(
    adata_path,
    result_dir,
    cell_chunk_size=ZERO_RATE_CELL_CHUNK_SIZE,
    range_hist_bins=ZERO_RATE_RANGE_HIST_BINS,
    plot_bins=ZERO_RATE_PLOT_BINS,
):
    # Pass 1: exact min/max of log10(lambda), streaming over all cell-gene pairs.
    log_min = np.inf
    log_max = -np.inf
    for _, _, log10_lam in iter_zip_log10_lambda_blocks(result_dir, cell_chunk_size):
        log_min = min(log_min, float(np.nanmin(log10_lam)))
        log_max = max(log_max, float(np.nanmax(log10_lam)))

    # Pass 2: all-pair histogram for percentile trimming without storing all values.
    range_edges = np.linspace(log_min, log_max, range_hist_bins + 1)
    range_hist = np.zeros(range_hist_bins, dtype=np.int64)
    for _, _, log10_lam in iter_zip_log10_lambda_blocks(result_dir, cell_chunk_size):
        range_hist += np.histogram(log10_lam, bins=range_edges)[0]

    lower = histogram_quantile(range_hist, range_edges, ZERO_RATE_LOWER_Q)
    upper = histogram_quantile(range_hist, range_edges, ZERO_RATE_UPPER_Q)
    plot_edges = np.linspace(lower, upper, plot_bins + 1)
    plot_centers = 0.5 * (plot_edges[:-1] + plot_edges[1:])

    total_counts = np.zeros(plot_bins, dtype=np.int64)
    nonzero_counts = np.zeros(plot_bins, dtype=np.int64)

    with h5py.File(adata_path, "r") as handle:
        x_group = handle["X"]
        shape = tuple(int(v) for v in x_group.attrs["shape"])
        n_obs, n_vars = shape
        indptr = x_group["indptr"][:]
        indices = x_group["indices"]

        for start, end, log10_lam in iter_zip_log10_lambda_blocks(result_dir, cell_chunk_size):
            if log10_lam.shape[1] != n_vars:
                raise ValueError("ZIP gene latent dimension does not match adata.X shape")

            scaled = (log10_lam - lower) / (upper - lower) * plot_bins
            bin_idx = np.floor(scaled).astype(np.int16, copy=False)
            valid = (bin_idx >= 0) & (bin_idx < plot_bins)
            total_counts += np.bincount(bin_idx[valid], minlength=plot_bins)

            for local_row, row in enumerate(range(start, end)):
                row_start = int(indptr[row])
                row_end = int(indptr[row + 1])
                if row_end == row_start:
                    continue
                row_cols = indices[row_start:row_end]
                row_bins = bin_idx[local_row, row_cols]
                row_bins = row_bins[(row_bins >= 0) & (row_bins < plot_bins)]
                if row_bins.size:
                    nonzero_counts += np.bincount(row_bins, minlength=plot_bins)

    zero_counts = total_counts - nonzero_counts
    observed_zero_rate = np.full(plot_bins, np.nan, dtype=np.float64)
    valid_bins = total_counts > 50
    observed_zero_rate[valid_bins] = zero_counts[valid_bins] / total_counts[valid_bins]

    return plot_centers, observed_zero_rate, total_counts, nonzero_counts


bin_centers, observed_zero_rate, zero_rate_bin_counts, zero_rate_nonzero_counts = full_matrix_zero_rate_by_zip_intensity(
    ADATA_PATH,
    ZIP_DIR,
)

eta_grid = bin_centers * np.log(10.0)
scene_zero_curve = 1.0 - sigmoid(alpha_zip * eta_grid)

fig, ax = plt.subplots(figsize=(3.4, 2.4))
ax.scatter(
    bin_centers,
    observed_zero_rate,
    s=12,
    alpha=0.8,
    color="0.25",
    label="Observed zero rate, binned",
    zorder=3,
)
ax.plot(
    bin_centers,
    scene_zero_curve,
    "--",
    lw=1.1,
    color="#D76155",
    label=rf"SCENE-ZIP: $1-\sigma(\alpha\eta)$, $\alpha={alpha_zip:.2f}$",
)
ax.set_xlabel(r"$\log_{10}(\lambda)$ predicted Poisson intensity")
ax.set_ylabel("Fraction of zeros")
ax.set_title("Detection vs intensity")
ax.set_ylim(-0.02, 1.05)
ax.legend(frameon=False)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "app_zero_rate_detection_vs_intensity.svg", dpi=600, bbox_inches="tight")
plt.show()

## Alpha across SCENE dimension sweeps and dataset sparsity

The learned ZIP scaling parameter `alpha` controls how sharply latent affinity is converted into nonzero detection probability. This summary uses only `SCENE_*D_seed*` runs, aggregates alpha across seeds for each latent dimension, and compares those values with the observed zero fraction of the model input matrix (`layers/counts` when present, otherwise `X`).


In [ ]:
import re

SCENE_ALPHA_DATASETS = {
    "PBMC CITE-seq": {
        "results_dir": PROJECT_ROOT / "results/pbmc_cite_seq",
        "adata_path": PROJECT_ROOT / "data/pbmc_cite_seq.h5ad",
        "color": "#4C9A8A",
    },
    "HCA nuclei": {
        "results_dir": PROJECT_ROOT / "results/hca_nuclei",
        "adata_path": PROJECT_ROOT / "data/hca_nuclei.h5ad",
        "color": "#D76155",
    },
    "Cortex": {
        "results_dir": PROJECT_ROOT / "results/cortex",
        "adata_path": PROJECT_ROOT / "data/cortex.h5ad",
        "color": "#6B7FD7",
    },
    "NeurIPS stem cells": {
        "results_dir": PROJECT_ROOT / "results/neurips_cite_stem_cells",
        "adata_path": PROJECT_ROOT / "data/neurips_cite_gex_stem_cells.h5ad",
        "color": "#B07AA1",
    },
}


def h5ad_model_matrix_sparsity(path, preferred_layer="counts", chunk_size=10_000_000):
    with h5py.File(path, "r") as handle:
        if preferred_layer in handle.get("layers", {}):
            matrix = handle["layers"][preferred_layer]
            matrix_source = f"layers/{preferred_layer}"
        else:
            matrix = handle["X"]
            matrix_source = "X"

        n_positive = 0
        encoding = matrix.attrs.get("encoding-type", "array")

        if encoding in {"csr_matrix", "csc_matrix"}:
            n_obs, n_vars = (int(v) for v in matrix.attrs["shape"])
            matrix_data = matrix["data"]
            for start in range(0, matrix_data.shape[0], chunk_size):
                values = matrix_data[start:start + chunk_size]
                n_positive += int(np.count_nonzero(values > 0))
        else:
            n_obs, n_vars = matrix.shape
            rows_per_chunk = max(1, min(n_obs, chunk_size // max(1, n_vars)))
            for start in range(0, n_obs, rows_per_chunk):
                values = matrix[start:start + rows_per_chunk]
                n_positive += int(np.count_nonzero(values > 0))

    n_total = n_obs * n_vars
    density = n_positive / n_total
    return {
        "n_obs": n_obs,
        "n_vars": n_vars,
        "n_positive": n_positive,
        "n_total": n_total,
        "density": density,
        "sparsity": 1.0 - density,
        "matrix_source": matrix_source,
    }


def collect_scene_alpha_runs(dataset_label, results_dir):
    rows = []
    run_pattern = re.compile(r"^SCENE_(\d+)D_seed(\d+)$")

    for params_path in sorted(results_dir.glob("SCENE_*D_seed*/scLDM_params.json")):
        run_name = params_path.parent.name
        match = run_pattern.match(run_name)
        if match is None:
            continue

        dim = int(match.group(1))
        seed = int(match.group(2))
        config = json.loads((params_path.parent / "config.json").read_text())
        params = json.loads(params_path.read_text())

        rows.append({
            "dataset": dataset_label,
            "run_name": run_name,
            "latent_dim": dim,
            "seed": seed,
            "config_latent_dim": int(config["latent_dim"]),
            "alpha": float(params["alpha"]),
        })

    return rows


alpha_rows = []
sparsity_rows = []
for dataset_label, dataset_info in SCENE_ALPHA_DATASETS.items():
    alpha_rows.extend(collect_scene_alpha_runs(dataset_label, dataset_info["results_dir"]))
    sparsity_rows.append({
        "dataset": dataset_label,
        **h5ad_model_matrix_sparsity(dataset_info["adata_path"]),
    })

alpha_df = pd.DataFrame(alpha_rows)
sparsity_df = pd.DataFrame(sparsity_rows)

if alpha_df.empty:
    raise ValueError("No SCENE_*D_seed* alpha runs found")
if not (alpha_df["latent_dim"] == alpha_df["config_latent_dim"]).all():
    raise ValueError("At least one run-name dimension disagrees with config latent_dim")

alpha_summary = (
    alpha_df
    .groupby(["dataset", "latent_dim"], as_index=False)
    .agg(
        alpha_mean=("alpha", "mean"),
        alpha_sd=("alpha", "std"),
        alpha_sem=("alpha", lambda x: x.std(ddof=1) / np.sqrt(x.shape[0])),
        n_seeds=("alpha", "size"),
    )
    .merge(sparsity_df[["dataset", "sparsity", "density", "matrix_source"]], on="dataset", how="left")
)

alpha_summary.to_csv(OUTPUT_DIR / "app_scene_alpha_by_dim_sparsity.csv", index=False)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(7.0, 2.6),
    gridspec_kw={"width_ratios": [2.1, 1.25]},
)

ax = axes[0]
dims = sorted(alpha_summary["latent_dim"].unique())
dim_positions = np.arange(len(dims))
dim_to_position = {dim: i for i, dim in enumerate(dims)}

for dataset_label, dataset_info in SCENE_ALPHA_DATASETS.items():
    subset = alpha_summary[alpha_summary["dataset"] == dataset_label].sort_values("latent_dim")
    x = subset["latent_dim"].map(dim_to_position).to_numpy()
    ax.errorbar(
        x,
        subset["alpha_mean"],
        yerr=subset["alpha_sd"],
        marker="o",
        markersize=3.2,
        linewidth=1.1,
        capsize=2,
        color=dataset_info["color"],
        label=dataset_label,
    )

ax.set_xticks(dim_positions)
ax.set_xticklabels([str(dim) for dim in dims])
ax.set_xlabel("Latent dimension")
ax.set_ylabel(r"Learned ZIP scale $\alpha$")
ax.set_title(r"SCENE $\alpha$ across dimensions")
ax.legend(frameon=False)

ax = axes[1]
sparsity_plot = sparsity_df.set_index("dataset").loc[list(SCENE_ALPHA_DATASETS)]
bar_colors = [SCENE_ALPHA_DATASETS[label]["color"] for label in sparsity_plot.index]
bars = ax.bar(
    np.arange(sparsity_plot.shape[0]),
    sparsity_plot["sparsity"],
    color=bar_colors,
    width=0.6,
)
ax.set_xticks(np.arange(sparsity_plot.shape[0]))
ax.set_xticklabels(sparsity_plot.index, rotation=25, ha="right")
ax.set_ylabel("Fraction zeros")
ax.set_title("Dataset sparsity")
ax.set_ylim(0, 1.1)

for bar, sparsity in zip(bars, sparsity_plot["sparsity"]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        sparsity + 0.02,
        f"{sparsity:.1%}",
        ha="center",
        va="bottom",
        fontsize=5.2,
    )

plt.tight_layout()
fig.savefig(OUTPUT_DIR / "app_scene_alpha_by_dim_sparsity.svg", dpi=600, bbox_inches="tight")
plt.show()

display(alpha_summary)


## Detection-boundary diagnostic

Compare the fitted latent intensity predictor `eta = log(lambda)` for sampled zero and nonzero cell-gene pairs, and overlay the learned detection curve `pi = sigmoid(alpha * eta)`. The default below uses the 16D seed-0 run for each dataset; change `ETA_DIAGNOSTIC_DIM` or `ETA_DIAGNOSTIC_SEED` to inspect another run.


In [ ]:
ETA_DIAGNOSTIC_DIM = 16
ETA_DIAGNOSTIC_SEED = 0
ETA_DIAGNOSTIC_PAIRS_PER_CLASS = 30_000
ETA_DIAGNOSTIC_RNG_SEED = 42
ETA_DIAGNOSTIC_CANDIDATE_BATCH = 250_000
ETA_DIAGNOSTIC_PAIR_BATCH = 20_000


def h5ad_axis_names(handle, axis_key):
    axis_group = handle[axis_key]
    index_key = axis_group.attrs.get("_index", "_index")
    if isinstance(index_key, bytes):
        index_key = index_key.decode()
    return axis_group[index_key].asstr()[()]


def h5ad_model_matrix(handle, preferred_layer="counts"):
    if preferred_layer in handle.get("layers", {}):
        return handle["layers"][preferred_layer], f"layers/{preferred_layer}"
    return handle["X"], "X"


def result_to_adata_index(result_names, adata_names, axis_label):
    adata_lookup = {str(name): i for i, name in enumerate(adata_names)}
    result_idx = np.empty(len(result_names), dtype=np.int64)
    missing = []

    for i, name in enumerate(result_names.astype(str)):
        idx = adata_lookup.get(str(name))
        if idx is None:
            missing.append(str(name))
            result_idx[i] = -1
        else:
            result_idx[i] = idx

    if missing:
        preview = ", ".join(missing[:5])
        raise ValueError(f"Missing {len(missing)} {axis_label} names in h5ad index; first missing: {preview}")

    return result_idx


def adata_to_result_index(result_to_adata, adata_size):
    inverse = np.full(adata_size, -1, dtype=np.int64)
    inverse[result_to_adata] = np.arange(result_to_adata.shape[0], dtype=np.int64)
    return inverse


def matrix_nonzero_lookup(matrix, adata_rows, adata_cols):
    adata_rows = np.asarray(adata_rows, dtype=np.int64)
    adata_cols = np.asarray(adata_cols, dtype=np.int64)
    out = np.zeros(adata_rows.shape[0], dtype=bool)
    encoding = matrix.attrs.get("encoding-type", "array")

    if encoding in {"csr_matrix", "csc_matrix"}:
        if encoding != "csr_matrix":
            raise ValueError("Pair lookup currently expects CSR matrices or dense arrays")

        indptr = matrix["indptr"]
        indices = matrix["indices"]
        data = matrix["data"]
        order = np.argsort(adata_rows, kind="mergesort")
        sorted_rows = adata_rows[order]
        sorted_cols = adata_cols[order]

        start = 0
        while start < sorted_rows.shape[0]:
            row = int(sorted_rows[start])
            end = start + 1
            while end < sorted_rows.shape[0] and sorted_rows[end] == row:
                end += 1

            row_start = int(indptr[row])
            row_end = int(indptr[row + 1])
            row_cols = indices[row_start:row_end]
            if row_cols.size:
                query_cols = sorted_cols[start:end]
                if row_cols.size > 1 and np.any(row_cols[1:] < row_cols[:-1]):
                    hits = np.isin(query_cols, row_cols)
                    if hits.any():
                        row_data = data[row_start:row_end]
                        value_lookup = {int(c): row_data[i] for i, c in enumerate(row_cols)}
                        hits = np.array([value_lookup.get(int(c), 0) > 0 for c in query_cols], dtype=bool)
                else:
                    pos = np.searchsorted(row_cols, query_cols)
                    hits = pos < row_cols.size
                    hits[hits] = row_cols[pos[hits]] == query_cols[hits]
                    if hits.any():
                        row_data = data[row_start:row_end]
                        hits[hits] = row_data[pos[hits]] > 0
                out[order[start:end]] = hits

            start = end

    else:
        order = np.argsort(adata_rows, kind="mergesort")
        sorted_rows = adata_rows[order]
        sorted_cols = adata_cols[order]
        start = 0
        while start < sorted_rows.shape[0]:
            row = int(sorted_rows[start])
            end = start + 1
            while end < sorted_rows.shape[0] and sorted_rows[end] == row:
                end += 1
            row_values = matrix[row, :]
            out[order[start:end]] = row_values[sorted_cols[start:end]] > 0
            start = end

    return out


def sample_pairs_by_status(matrix, result_to_adata_obs, result_to_adata_var, want_nonzero, n_pairs, rng):
    n_obs = result_to_adata_obs.shape[0]
    n_vars = result_to_adata_var.shape[0]
    sampled_rows = []
    sampled_cols = []
    n_collected = 0
    attempts = 0
    max_attempts = 100

    while n_collected < n_pairs and attempts < max_attempts:
        attempts += 1
        candidate_n = max(ETA_DIAGNOSTIC_CANDIDATE_BATCH, (n_pairs - n_collected) * 5)
        rows = rng.integers(0, n_obs, size=candidate_n, dtype=np.int64)
        cols = rng.integers(0, n_vars, size=candidate_n, dtype=np.int64)
        adata_rows = result_to_adata_obs[rows]
        adata_cols = result_to_adata_var[cols]

        is_nonzero = matrix_nonzero_lookup(matrix, adata_rows, adata_cols)
        keep = is_nonzero if want_nonzero else ~is_nonzero
        if not keep.any():
            continue

        rows = rows[keep]
        cols = cols[keep]
        needed = n_pairs - n_collected
        if rows.shape[0] > needed:
            rows = rows[:needed]
            cols = cols[:needed]

        sampled_rows.append(rows)
        sampled_cols.append(cols)
        n_collected += rows.shape[0]

    if n_collected < n_pairs:
        raise ValueError(f"Only sampled {n_collected:,} pairs for status want_nonzero={want_nonzero}")

    return np.concatenate(sampled_rows), np.concatenate(sampled_cols)


def h5ad_categorical_codes_for_result_cells(handle, key, result_to_adata_obs, expected_categories):
    obs_group = handle["obs"][key]
    codes = obs_group["codes"][:]
    categories = obs_group["categories"].asstr()[()]
    category_to_expected = {str(cat): i for i, cat in enumerate(expected_categories)}

    mapped = np.full(categories.shape[0], -1, dtype=np.int64)
    for i, category in enumerate(categories):
        if str(category) in category_to_expected:
            mapped[i] = category_to_expected[str(category)]

    result_codes = mapped[codes[result_to_adata_obs]]
    if np.any(result_codes < 0):
        raise ValueError(f"Could not map all categories for batch key {key}")
    return result_codes


def load_result_batch_effects(result_dir, params, handle, result_to_adata_obs):
    batch_levels = list(params.get("batch_levels", []))
    gamma_levels = list(params.get("gamma_levels", []))
    if not batch_levels:
        return []

    batch_npz_path = result_dir / "scLDM_batch_effects.npz"
    if not batch_npz_path.exists():
        return []

    effects = np.load(batch_npz_path)
    loaded = []
    for level_idx, batch_key in enumerate(batch_levels):
        tag = gamma_levels[level_idx]["tag"] if level_idx < len(gamma_levels) else params["variant_levels"][level_idx]
        batch_ids = h5ad_categorical_codes_for_result_cells(
            handle,
            batch_key,
            result_to_adata_obs,
            params["batch_categories"][batch_key],
        )
        level_key = f"level_{level_idx}"
        if tag == "full":
            loaded.append({
                "tag": "full",
                "batch_ids": batch_ids,
                "gamma": effects[f"{level_key}__gamma"],
            })
        elif tag == "lowrank":
            loaded.append({
                "tag": "lowrank",
                "batch_ids": batch_ids,
                "U": effects[f"{level_key}__U"],
                "V": effects[f"{level_key}__V"],
            })
        elif tag == "none":
            continue
        else:
            raise ValueError(f"Unknown batch effect tag: {tag}")

    return loaded


def compute_eta_for_pairs(result_dir, rows, cols, batch_effects=None, pair_batch_size=ETA_DIAGNOSTIC_PAIR_BATCH):
    z_cells = np.load(result_dir / "scLDM_cell_latent.npy", mmap_mode="r")
    z_genes = np.load(result_dir / "scLDM_gene_latent.npy", mmap_mode="r")
    re_cells = np.load(result_dir / "scLDM_re_cell.npy", mmap_mode="r").reshape(-1)
    re_genes = np.load(result_dir / "scLDM_re_gene.npy", mmap_mode="r").reshape(-1)

    rows = np.asarray(rows, dtype=np.int64)
    cols = np.asarray(cols, dtype=np.int64)
    eta = np.empty(rows.shape[0], dtype=np.float32)
    batch_effects = batch_effects or []

    for start in range(0, rows.shape[0], pair_batch_size):
        end = min(start + pair_batch_size, rows.shape[0])
        r = rows[start:end]
        c = cols[start:end]
        zc = np.asarray(z_cells[r], dtype=np.float32)
        zg = np.asarray(z_genes[c], dtype=np.float32)
        values = re_cells[r].astype(np.float32) + re_genes[c].astype(np.float32)
        values -= np.linalg.norm(zc - zg, axis=1).astype(np.float32)

        for effect in batch_effects:
            batch_ids = effect["batch_ids"][r]
            if effect["tag"] == "full":
                values += effect["gamma"][c, batch_ids]
            elif effect["tag"] == "lowrank":
                values += np.sum(effect["U"][batch_ids] * effect["V"][c], axis=1)

        eta[start:end] = values

    return eta


def collect_eta_detection_diagnostic(dataset_label, dataset_info, dim, seed, n_pairs, rng):
    result_dir = dataset_info["results_dir"] / f"SCENE_{dim}D_seed{seed}"
    params = json.loads((result_dir / "scLDM_params.json").read_text())
    result_cell_names = np.load(result_dir / "scLDM_cell_names.npy", allow_pickle=True).astype(str)
    result_gene_names = np.load(result_dir / "scLDM_gene_names.npy", allow_pickle=True).astype(str)

    with h5py.File(dataset_info["adata_path"], "r") as handle:
        matrix, matrix_source = h5ad_model_matrix(handle)
        obs_names = h5ad_axis_names(handle, "obs")
        var_names = h5ad_axis_names(handle, "var")
        result_to_adata_obs = result_to_adata_index(result_cell_names, obs_names, "cell")
        result_to_adata_var = result_to_adata_index(result_gene_names, var_names, "gene")

        zero_rows, zero_cols = sample_pairs_by_status(
            matrix,
            result_to_adata_obs,
            result_to_adata_var,
            want_nonzero=False,
            n_pairs=n_pairs,
            rng=rng,
        )
        nonzero_rows, nonzero_cols = sample_pairs_by_status(
            matrix,
            result_to_adata_obs,
            result_to_adata_var,
            want_nonzero=True,
            n_pairs=n_pairs,
            rng=rng,
        )
        batch_effects = load_result_batch_effects(result_dir, params, handle, result_to_adata_obs)

    eta_zero = compute_eta_for_pairs(result_dir, zero_rows, zero_cols, batch_effects=batch_effects)
    eta_nonzero = compute_eta_for_pairs(result_dir, nonzero_rows, nonzero_cols, batch_effects=batch_effects)

    return {
        "dataset": dataset_label,
        "matrix_source": matrix_source,
        "alpha": float(params["alpha"]),
        "eta_zero": eta_zero,
        "eta_nonzero": eta_nonzero,
    }


eta_rng = np.random.default_rng(ETA_DIAGNOSTIC_RNG_SEED)
eta_diagnostics = [
    collect_eta_detection_diagnostic(
        dataset_label,
        dataset_info,
        ETA_DIAGNOSTIC_DIM,
        ETA_DIAGNOSTIC_SEED,
        ETA_DIAGNOSTIC_PAIRS_PER_CLASS,
        eta_rng,
    )
    for dataset_label, dataset_info in SCENE_ALPHA_DATASETS.items()
]

eta_summary_rows = []
for diagnostic in eta_diagnostics:
    for status, values in [("zero", diagnostic["eta_zero"]), ("nonzero", diagnostic["eta_nonzero"],)]:
        eta_summary_rows.append({
            "dataset": diagnostic["dataset"],
            "status": status,
            "dim": ETA_DIAGNOSTIC_DIM,
            "seed": ETA_DIAGNOSTIC_SEED,
            "alpha": diagnostic["alpha"],
            "matrix_source": diagnostic["matrix_source"],
            "eta_mean": float(np.mean(values)),
            "eta_median": float(np.median(values)),
            "eta_q10": float(np.quantile(values, 0.10)),
            "eta_q90": float(np.quantile(values, 0.90)),
        })

eta_summary = pd.DataFrame(eta_summary_rows)

fig, axes = plt.subplots(2, 2, figsize=(7.0, 4.8), sharex=False, sharey=False)
axes = axes.ravel()
zero_color = "0.35"
nonzero_color = "#D76155"
curve_color = "#2B6CB0"
all_eta_values = np.concatenate(
    [diagnostic["eta_zero"] for diagnostic in eta_diagnostics]
    + [diagnostic["eta_nonzero"] for diagnostic in eta_diagnostics]
)
eta_xlim = np.quantile(all_eta_values[np.isfinite(all_eta_values)], [0.005, 0.999])
eta_lo, eta_hi = float(eta_xlim[0]), float(eta_xlim[1])
if not np.isfinite(eta_lo) or not np.isfinite(eta_hi) or eta_lo == eta_hi:
    eta_lo = float(np.nanmin(all_eta_values))
    eta_hi = float(np.nanmax(all_eta_values))
eta_bins = np.linspace(eta_lo, eta_hi, 45)

def histogram_overlap(values_a, values_b, bins):
    counts_a, _ = np.histogram(values_a, bins=bins)
    counts_b, _ = np.histogram(values_b, bins=bins)
    if counts_a.sum() == 0 or counts_b.sum() == 0:
        return np.nan
    prob_a = counts_a / counts_a.sum()
    prob_b = counts_b / counts_b.sum()
    return float(np.minimum(prob_a, prob_b).sum())

eta_overlap_by_dataset = {
    diagnostic["dataset"]: histogram_overlap(diagnostic["eta_zero"], diagnostic["eta_nonzero"], eta_bins)
    for diagnostic in eta_diagnostics
}
eta_summary["eta_overlap_zero_nonzero"] = eta_summary["dataset"].map(eta_overlap_by_dataset)
eta_summary["eta_separation_one_minus_overlap"] = 1.0 - eta_summary["eta_overlap_zero_nonzero"]
eta_summary.to_csv(OUTPUT_DIR / "app_scene_eta_zero_nonzero_summary.csv", index=False)

eta_density_max = 0.0
for diagnostic in eta_diagnostics:
    zero_density, _ = np.histogram(diagnostic["eta_zero"], bins=eta_bins, density=True)
    nonzero_density, _ = np.histogram(diagnostic["eta_nonzero"], bins=eta_bins, density=True)
    eta_density_max = max(
        eta_density_max,
        float(np.nanmax(zero_density)),
        float(np.nanmax(nonzero_density)),
    )
eta_density_ylim = eta_density_max * 1.08

for ax, diagnostic in zip(axes, eta_diagnostics):
    eta_zero = diagnostic["eta_zero"]
    eta_nonzero = diagnostic["eta_nonzero"]
    alpha = diagnostic["alpha"]
    dataset_label = diagnostic["dataset"]
    zero_median = float(np.median(eta_zero))
    nonzero_median = float(np.median(eta_nonzero))
    eta_overlap = eta_overlap_by_dataset[dataset_label]

    grid = np.linspace(eta_lo, eta_hi, 300)

    ax.hist(
        eta_zero,
        bins=eta_bins,
        density=True,
        histtype="stepfilled",
        alpha=0.35,
        color=zero_color,
        label="Observed zero",
    )
    ax.hist(
        eta_nonzero,
        bins=eta_bins,
        density=True,
        histtype="step",
        linewidth=1.2,
        color=nonzero_color,
        label="Observed nonzero",
    )
    ax.axvline(zero_median, color=zero_color, lw=0.9, ls=":", label="Zero median")
    ax.axvline(nonzero_median, color=nonzero_color, lw=0.9, ls=":", label="Nonzero median")
    ax.set_xlim(eta_lo, eta_hi)
    ax.set_ylim(0, eta_density_ylim)
    ax.text(
        0.98,
        0.93,
        f"overlap={eta_overlap:.2f}",
        transform=ax.transAxes,
        ha="right",
        va="top",
        fontsize=5.5,
        bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.75, "pad": 1.5},
    )
    ax.set_title(f"{dataset_label}\n{ETA_DIAGNOSTIC_DIM}D seed {ETA_DIAGNOSTIC_SEED}, alpha={alpha:.2f}")
    ax.set_xlabel(r"$\eta = \log(\lambda)$")
    ax.set_ylabel("Density")

    ax_pi = ax.twinx()
    ax_pi.plot(grid, 1.0 / (1.0 + np.exp(-alpha * grid)), color=curve_color, lw=1.0, ls="--")
    ax_pi.set_ylim(-0.02, 1.02)
    ax_pi.tick_params(axis="y", labelsize=5.2, colors=curve_color)
    ax_pi.spines["right"].set_color(curve_color)
    if ax is axes[-1]:
        ax_pi.set_ylabel(r"$\pi = \sigma(\alpha\eta)$", color=curve_color)
    else:
        ax_pi.set_yticklabels([])

axes[0].legend(frameon=False, loc="upper left")
fig.suptitle("Zero/nonzero separation in fitted SCENE intensity", y=1.01, fontsize=8)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "app_scene_eta_zero_nonzero_diagnostic.svg", dpi=600, bbox_inches="tight")
plt.show()

display(eta_summary)


## Zero-rate vs intensity across SCENE datasets

Sample cell-gene pairs from the same `SCENE_16D_seed0` run family used above, bin them by fitted intensity, and compare the empirical zero fraction with the learned ZIP detection curve. This is the multi-dataset version of `app_zero_rate_detection_vs_intensity.svg`.


In [ ]:
ZERO_RATE_MULTI_DIM = ETA_DIAGNOSTIC_DIM
ZERO_RATE_MULTI_SEED = ETA_DIAGNOSTIC_SEED
ZERO_RATE_MULTI_SAMPLE_PAIRS = 500_000
ZERO_RATE_MULTI_BINS = 40
ZERO_RATE_MULTI_LOG10_RANGE = (-3.0, 3.0)
ZERO_RATE_MULTI_MIN_BIN_PAIRS = 50
ZERO_RATE_MULTI_RNG_SEED = 123
ZERO_RATE_MULTI_PAIR_BATCH = ETA_DIAGNOSTIC_PAIR_BATCH


def collect_sampled_zero_rate_by_intensity(
    dataset_label,
    dataset_info,
    dim=ZERO_RATE_MULTI_DIM,
    seed=ZERO_RATE_MULTI_SEED,
    n_pairs=ZERO_RATE_MULTI_SAMPLE_PAIRS,
    rng=None,
):
    rng = np.random.default_rng(ZERO_RATE_MULTI_RNG_SEED) if rng is None else rng
    result_dir = dataset_info["results_dir"] / f"SCENE_{dim}D_seed{seed}"
    params = json.loads((result_dir / "scLDM_params.json").read_text())
    result_cell_names = np.load(result_dir / "scLDM_cell_names.npy", allow_pickle=True).astype(str)
    result_gene_names = np.load(result_dir / "scLDM_gene_names.npy", allow_pickle=True).astype(str)

    with h5py.File(dataset_info["adata_path"], "r") as handle:
        matrix, matrix_source = h5ad_model_matrix(handle)
        obs_names = h5ad_axis_names(handle, "obs")
        var_names = h5ad_axis_names(handle, "var")
        result_to_adata_obs = result_to_adata_index(result_cell_names, obs_names, "cell")
        result_to_adata_var = result_to_adata_index(result_gene_names, var_names, "gene")

        rows = rng.integers(0, result_to_adata_obs.shape[0], size=n_pairs, dtype=np.int64)
        cols = rng.integers(0, result_to_adata_var.shape[0], size=n_pairs, dtype=np.int64)
        is_nonzero = matrix_nonzero_lookup(
            matrix,
            result_to_adata_obs[rows],
            result_to_adata_var[cols],
        )
        batch_effects = load_result_batch_effects(result_dir, params, handle, result_to_adata_obs)

    eta = compute_eta_for_pairs(
        result_dir,
        rows,
        cols,
        batch_effects=batch_effects,
        pair_batch_size=ZERO_RATE_MULTI_PAIR_BATCH,
    )
    log10_lambda = eta / np.log(10.0)

    return {
        "dataset": dataset_label,
        "matrix_source": matrix_source,
        "alpha": float(params["alpha"]),
        "log10_lambda": log10_lambda,
        "is_zero": ~is_nonzero,
        "n_pairs": int(n_pairs),
    }


zero_rate_rng = np.random.default_rng(ZERO_RATE_MULTI_RNG_SEED)
zero_rate_diagnostics = [
    collect_sampled_zero_rate_by_intensity(
        dataset_label,
        dataset_info,
        rng=zero_rate_rng,
    )
    for dataset_label, dataset_info in SCENE_ALPHA_DATASETS.items()
]

log10_lo, log10_hi = ZERO_RATE_MULTI_LOG10_RANGE
zero_rate_edges = np.linspace(log10_lo, log10_hi, ZERO_RATE_MULTI_BINS + 1)
zero_rate_centers = 0.5 * (zero_rate_edges[:-1] + zero_rate_edges[1:])
zero_rate_rows = []

for diagnostic in zero_rate_diagnostics:
    log10_lambda = diagnostic["log10_lambda"]
    is_zero = diagnostic["is_zero"]
    in_range = (
        np.isfinite(log10_lambda)
        & (log10_lambda > log10_lo)
        & (log10_lambda < log10_hi)
    )
    bin_idx = np.digitize(log10_lambda[in_range], zero_rate_edges) - 1
    bin_idx = np.clip(bin_idx, 0, ZERO_RATE_MULTI_BINS - 1)
    zero_values = is_zero[in_range].astype(float)

    total_counts = np.bincount(bin_idx, minlength=ZERO_RATE_MULTI_BINS)
    zero_counts = np.bincount(
        bin_idx,
        weights=zero_values,
        minlength=ZERO_RATE_MULTI_BINS,
    )

    with np.errstate(divide="ignore", invalid="ignore"):
        observed_zero_rate = zero_counts / total_counts
    observed_zero_rate[total_counts <= ZERO_RATE_MULTI_MIN_BIN_PAIRS] = np.nan

    for left, center, right, total, zero_count, zero_rate in zip(
        zero_rate_edges[:-1],
        zero_rate_centers,
        zero_rate_edges[1:],
        total_counts,
        zero_counts,
        observed_zero_rate,
    ):
        zero_rate_rows.append({
            "dataset": diagnostic["dataset"],
            "dim": ZERO_RATE_MULTI_DIM,
            "seed": ZERO_RATE_MULTI_SEED,
            "alpha": diagnostic["alpha"],
            "matrix_source": diagnostic["matrix_source"],
            "log10_lambda_bin_left": float(left),
            "log10_lambda_bin_center": float(center),
            "log10_lambda_bin_right": float(right),
            "n_sampled_pairs": int(total),
            "n_zero_pairs": int(zero_count),
            "observed_zero_rate": float(zero_rate) if np.isfinite(zero_rate) else np.nan,
        })

zero_rate_df = pd.DataFrame(zero_rate_rows)
zero_rate_df.to_csv(OUTPUT_DIR / "app_zero_rate_detection_vs_intensity_by_dataset.csv", index=False)

fig, axes = plt.subplots(2, 2, figsize=(7, 4.8), sharex=True, sharey=True)
axes = axes.ravel()
curve_grid = np.linspace(log10_lo, log10_hi, 300)
eta_grid = curve_grid * np.log(10.0)

for ax, diagnostic in zip(axes, zero_rate_diagnostics):
    dataset_label = diagnostic["dataset"]
    point_color = SCENE_ALPHA_DATASETS[dataset_label]["color"]
    dataset_rows = zero_rate_df[zero_rate_df["dataset"] == dataset_label]
    plot_rows = dataset_rows.dropna(subset=["observed_zero_rate"])
    alpha = diagnostic["alpha"]
    scene_zero_curve = 1.0 - sigmoid(alpha * eta_grid)

    ax.scatter(
        plot_rows["log10_lambda_bin_center"],
        plot_rows["observed_zero_rate"],
        s=12,
        alpha=0.8,
        color=point_color,
        label="Empirical (binned)",
        zorder=3,
    )
    ax.plot(
        curve_grid,
        scene_zero_curve,
        "--",
        lw=1.2,
        color="0.15",
        label=rf"Model: single $\alpha$",
    )
    ax.set_title(f"{dataset_label}\nSCENE {ZERO_RATE_MULTI_DIM}D seed {ZERO_RATE_MULTI_SEED}, MAP $\\alpha$={alpha:.2f}")
    ax.set_xlim(log10_lo, log10_hi)
    ax.set_ylim(-0.02, 1.05)
    ax.set_xlabel(r"$\log_{10}(\lambda)$  (Poisson intensity)")
    ax.set_ylabel("Fraction of zeros")

legend_handles = [
    mpl.lines.Line2D([], [], marker="o", linestyle="none", color="0.45", markersize=4, label="Empirical (binned)"),
    mpl.lines.Line2D([], [], linestyle="--", color="0.15", linewidth=1.2, label=rf"Model: single $\alpha$"),
]
fig.legend(handles=legend_handles, frameon=False, loc="upper center", bbox_to_anchor=(0.5, 0.965), ncol=2)
fig.suptitle(r"Does a single $\alpha$ capture detection vs fitted intensity?", y=1.03, fontsize=8)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "app_zero_rate_detection_vs_intensity.svg", dpi=600, bbox_inches="tight")
plt.show()

display(zero_rate_df)
